# اجرای پژوهشی پایان‌نامه در Google Colab

**نسخهٔ اصلاح ۱۴۰۵/۰۶/۲۵:** بخش ۶ خودکفا شد (تعریف `cfg` و `CHECKPOINT` داخل سلول) + پرچم `USE_APPENDIX_PROFILE` برای پروفایل مستند پیوست + پیام دقیق پارامترهای جاافتاده + بهبود NIST (بخش ۱۰). اگر این خط را در نوت‌بوک بازشده در کولب نمی‌بینید، نسخهٔ قدیمی است؛ فایل تازه را از `published/scripts/medsec_colab/Thesis_Colab.ipynb` آپلود کنید.

ابتدا README_FA و CONFORMANCE_FA را بخوانید. آموزش و آزمون با دادهٔ واقعی انجام می‌شود؛ خروجی ساختگی نداریم. هفت پارامتر معادلهٔ اصلی نامشخص و دستگاه پیوست متفاوت و مستعد واگرایی است. هیچ عدد فصل چهارم از پیش تأیید نشده است. این بسته برای دادهٔ شناسایی‌پذیر بیمار یا استفادهٔ بالینی نیست.

In [ ]:
%pip install -q --upgrade kagglehub

## ۱. بارگذاری ZIP بسته
فایل `medical_thesis_colab.zip` را انتخاب کنید؛ این فایل فقط شامل کد و مستندات است. نوت‌بوک قدیمی اجرا نمی‌شود.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, subprocess
from google.colab import files
uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.endswith('.zip')]
if len(archives) != 1: raise ValueError('دقیقاً ZIP بسته را انتخاب کنید')
ROOT = Path('/content/thesis_toolkit')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as z:
    for name in z.namelist():
        if not (ROOT/name).resolve().is_relative_to(ROOT.resolve()): raise ValueError('Unsafe zip')
    z.extractall(ROOT)
PACKAGE = ROOT/'medsec_colab'
os.chdir(PACKAGE)
sys.path.insert(0, str(PACKAGE))
print('مسیر بسته:', PACKAGE)
version_file = PACKAGE/'PACKAGE_VERSION.txt'
if version_file.is_file():
    print('نسخهٔ بسته:', version_file.read_text(encoding='utf-8').strip().splitlines()[0])
else:
    print('هشدار: بستهٔ آپلودشده قدیمی است؛ medsec_colab_fixed.zip جدید را آپلود کنید.')

## ۲. نصب و آزمون نرم‌افزار
آزمون‌های خودکار از دادهٔ کوچک مصنوعی **فقط برای بررسی نرم‌افزار** استفاده می‌کنند؛ آن‌ها آزمایش پزشکی یا بازتولید نتایج نیستند.

In [ ]:
subprocess.run([sys.executable, 'scripts/install_colab.py'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', 'scripts/tests', '-q'], check=True)
from scripts.artifacts import environment
print(json.dumps(environment(), indent=2, ensure_ascii=False))

{
  "utc": "2026-09-15T18:27:34.519356+00:00",
  "python": "3.13.15",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "processor": "x86_64",
  "gpu": null,
  "packages": {
    "numpy": "2.1.3",
    "scipy": "1.16.3",
    "torch": "2.11.0+cpu",
    "numba": "0.61.2",
    "scikit-image": "0.25.2",
    "Pillow": "11.3.0",
    "nibabel": "5.4.2",
    "cryptography": "50.0.1"
  },
  "source_sha256": {
    "__init__.py": "ca0d97e51702b28ef3e3814d44d3591e365ae903a24905582728875e60038ea7",
    "__main__.py": "159df03237b3f6dce38f66b78397fbb427921e577061fd90f256ca3821154a6b",
    "artifacts.py": "72329cf074062eb81dcb8d43226a5cb650bbcac67e070b689b7c3ce03cb93ad5",
    "attacks.py": "77dc3e3ebb5d8345353c51bab5df37c9abf7fb99b449c71e608484bbf088aff3",
    "build_delivery.py": "c878097e31cdfacead428bbd40fed0b9bc8aafba2ab03a35d77d43cf52bbe4e8",
    "chaos.py": "4ec83407436b4c08dc7ccbe8f8a4a4e825347fed3bc2a5f673131277ac16142f",
    "cipher.py": "2e657a9c32f9793aa47eebafb550183648b75f708934e6b50

## ۳. مسیرهای ماندگار و تنظیمات
نام/نسخهٔ دقیق داده و مسیر واقعی آن را وارد کنید. مسیر WORK برای وزن و گزارش است؛ DATA_ROOT باید پوشهٔ داده‌های دریافت‌شده با مجوز باشد. برای هر دیتاست جدا اجرا کنید.

In [ ]:
MOUNT_DRIVE = True
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
os.environ['THESIS_WORKDIR'] = '/content/drive/MyDrive/Thesis_Research' if MOUNT_DRIVE else '/content/Thesis_Research'
WORK = Path(os.environ['THESIS_WORKDIR']); WORK.mkdir(parents=True, exist_ok=True)
DATASET = 'DRIVE'  # DRIVE / RITE / BraTS2020 / COVID19_CXR
DATA_ROOT = WORK/'data_sources'/DATASET
SOURCE = ''  # شناسه/نسخه و منشأ واقعی داده؛ خالی مجاز نیست
CXR_CSV = WORK/'cxr_index.csv'
MANIFEST = WORK/f'{DATASET}.jsonl'
WEIGHTS = WORK/'weights'/DATASET
PAYLOAD = WORK/'metadata.bin'  # فایل آزمایشی فاقد اطلاعات هویتی بیمار
RUN_DIR = WORK/'runs'/f'{DATASET}_001'  # در اجرای جدید نام تازه انتخاب کنید
PROFILE = PACKAGE/'scripts/thesis_reference.json'  # نسخه مستند تکمیل‌شده خودتان را جایگزین کنید
RUN_TRAINING = False
RESUME_TRAINING = False
RUN_EXPERIMENTS = False
RUN_NIST = False
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

Mounted at /content/drive
device: cpu


In [ ]:
"""Paste this whole file in one Colab cell AFTER the Drive/path configuration cell.

Downloads a pinned public REDISTRIBUTION, not an official-author archive.
Does not train, alter source code, fabricate masks, or create a research manifest.
"""
import hashlib
import json
import os
import shutil
import stat
import tempfile
import zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from urllib.request import Request, urlopen

from PIL import Image

# Public source configuration; terms remain those of the dataset providers.
os.environ['DRIVE_SOURCE_PAGE'] = 'https://www.kaggle.com/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction'
os.environ['DRIVE_OFFICIAL_PAGE'] = 'https://drive.grand-challenge.org/DRIVE/'
os.environ['DRIVE_ARCHIVE_URL'] = 'https://www.kaggle.com/api/v1/datasets/download/andrewmvd/drive-digital-retinal-images-for-vessel-extraction?datasetVersionNumber=1'
EXPECTED_SHA256 = '3efa1bf264da71a9080c9959c5ee89e2646193892efe9d66f29a4123b74f949a'


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def download_archive(destination):
    if not destination.exists():
        partial = destination.with_suffix('.zip.part')
        try:
            request = Request(os.environ['DRIVE_ARCHIVE_URL'], headers={'User-Agent': 'ThesisResearch/1.0'})
            with urlopen(request, timeout=90) as response, partial.open('wb') as f:
                total = 0
                while block := response.read(1024 * 1024):
                    total += len(block)
                    if total > 64 * 1024 * 1024:
                        raise ValueError('حجم پاسخ غیرمنتظره است؛ دریافت متوقف شد.')
                    f.write(block)
                    print(f'دریافت: {total / 1024**2:.1f} MiB', end='\r', flush=True)
            if file_hash(partial) != EXPECTED_SHA256:
                raise ValueError('هش دانلود مطابقت ندارد؛ فایل تغییرکرده، ناقص یا صفحهٔ ورود است.')
            partial.replace(destination)
            print()
        except Exception:
            partial.unlink(missing_ok=True)
            raise
    if file_hash(destination) != EXPECTED_SHA256:
        raise ValueError('آرشیو موجود با نسخهٔ بررسی‌شده متفاوت است؛ بازنویسی نشد.')


def unpack_verified(archive, destination):
    # Validate all members before extraction; never merge into existing user data.
    with zipfile.ZipFile(archive) as z:
        if sum(i.file_size for i in z.infolist()) > 128 * 1024 * 1024:
            raise ValueError('حجم استخراج غیرمنتظره است.')
        for info in z.infolist():
            p = PurePosixPath(info.filename)
            if (p.is_absolute() or '..' in p.parts or '\\' in info.filename
                    or not p.parts or p.parts[0] != 'DRIVE'
                    or stat.S_ISLNK(info.external_attr >> 16)):
                raise ValueError('مسیر غیرمجاز در ZIP')
        if z.testzip() is not None:
            raise ValueError('خرابی CRC آرشیو')
        if destination.exists():
            expected = {str(PurePosixPath(i.filename).relative_to('DRIVE')) for i in z.infolist() if not i.is_dir()}
            present = {str(p.relative_to(destination)) for p in destination.rglob('*') if p.is_file()}
            if expected != present:
                raise ValueError('پوشهٔ DRIVE قبلی متفاوت/ناقص است؛ هیچ فایلی بازنویسی نشد.')
            for info in z.infolist():
                if not info.is_dir():
                    p = destination / PurePosixPath(info.filename).relative_to('DRIVE')
                    if p.is_symlink() or file_hash(p) != hashlib.sha256(z.read(info)).hexdigest():
                        raise ValueError('محتوای DRIVE قبلی متفاوت است؛ بازنویسی نشد: ' + str(p))
        else:
            with tempfile.TemporaryDirectory(prefix='.drive-stage-', dir=destination.parent) as tmp:
                z.extractall(tmp)
                (Path(tmp) / 'DRIVE').rename(destination)


def inventory(root):
    counts = {}
    for split, ids in [('training', range(21, 41)), ('test', range(1, 21))]:
        images = sorted((root / split / 'images').glob('*.tif'))
        if {int(p.stem.split('_')[0]) for p in images} != set(ids) or len(images) != 20:
            raise ValueError('تعداد یا شناسهٔ تصاویر غیرمنتظره: ' + split)
        for path in images:
            with Image.open(path) as im:
                im.load()
                if im.size != (565, 584):
                    raise ValueError('ابعاد غیرمنتظره: ' + str(path))
        masks = sorted((root / split / '1st_manual').glob('*.gif'))
        if split == 'training' and ({int(p.stem.split('_')[0]) for p in masks} != set(ids) or len(masks) != 20):
            raise ValueError('ماسک دستی آموزش ناقص است.')
        for path in masks:
            with Image.open(path) as im:
                im.load()
                if im.size != (565, 584) or set(im.convert('L').tobytes()) != {0, 255}:
                    raise ValueError('ماسک مرجع نامعتبر: ' + str(path))
        counts[split] = {'images': len(images), 'vessel_manual_masks': len(masks)}
    return counts


def main():
    if not os.environ.get('THESIS_WORKDIR'):
        raise RuntimeError('ابتدا سلول ۳، اتصال Drive و تنظیم THESIS_WORKDIR را اجرا کنید.')
    work = Path(os.environ['THESIS_WORKDIR']).resolve()
    mount = Path('/content/drive')
    if not (mount / 'MyDrive').is_dir() or not work.is_relative_to((mount / 'MyDrive').resolve()):
        raise RuntimeError('این گام باید در Drive ماندگار اجرا شود؛ سلول ۳ را با MOUNT_DRIVE=True اجرا کنید.')
    source = work / 'data_sources' / 'DRIVE'
    downloads = work / 'downloads'
    downloads.mkdir(parents=True, exist_ok=True)
    source.parent.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(work).free < 200 * 1024 * 1024:
        raise OSError('فضای آزاد قابل‌مشاهده کافی نیست؛ سهمیهٔ خود Google Drive را نیز بررسی کنید.')
    print('منبع: بازنشر عمومی Kaggle، نسخهٔ ۱؛ نه آرشیو منتشرشده مستقیم توسط مؤلفان.')
    archive = downloads / 'DRIVE-kaggle-v1.zip'
    if archive.is_file() and source.is_dir():
        print('DRIVE از قبل موجود و کامل است؛ دانلود و استخراج تکراری انجام نشد.')
    else:
        print('DRIVE یافت نشد؛ دانلود و استخراج آغاز می‌شود.')
        download_archive(archive)
        unpack_verified(archive, source)
    counts = inventory(source)
    report = {
        'step': '01_download_drive', 'status': 'download_verified',
        'utc': datetime.now(timezone.utc).isoformat(), 'data_root': str(source),
        'source_page': os.environ['DRIVE_SOURCE_PAGE'], 'official_page': os.environ['DRIVE_OFFICIAL_PAGE'],
        'archive_url': os.environ['DRIVE_ARCHIVE_URL'], 'distribution': 'Kaggle redistribution; version 1',
        'sha256': file_hash(archive), 'archive_bytes': archive.stat().st_size, 'counts': counts,
        'checksum_scope': 'Integrity against archive inspected by assistant; NOT independent author authenticity verification',
        'test_vessel_ground_truth_available': counts['test']['vessel_manual_masks'] == 20,
        'warning': 'test/mask is FIELD OF VIEW, NOT vessel ground truth. No test Dice/IoU possible with this archive.',
        'training_executed': False, 'research_manifest_created': False,
    }
    report_path = work / 'drive_download_report.json'
    temporary = report_path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(report_path)
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print('گام ۱ تمام شد. فعلاً سلول ۴ و سلول‌های بعدی قبلی را اجرا نکنید.')
    return report


if __name__ == '__main__':
    STEP1_REPORT = main()

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from importlib.metadata import version
import os
import json
import hashlib

if "WORK" not in globals():
    raise RuntimeError("ابتدا سلول ۳، تنظیمات مسیرها، را اجرا کنید.")

if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("ابتدا Google Drive را متصل کنید.")

work = Path(WORK)
cache_root = work / "data_sources" / "_kaggle_cache"
os.environ["KAGGLEHUB_CACHE"] = str(cache_root)
os.environ["DISABLE_COLAB_CACHE"] = "true"

import kagglehub
from kagglehub import config as kagglehub_config

print(
    "کش اختصاصی کولب غیرفعال است؟",
    kagglehub_config.is_colab_cache_disabled()
)
print("مسیر ماندگار دانلود:", cache_root)

catalog = {
    "CHASE_DB1": "namnguynnnn/chase-db1/versions/1",
    "STARE": "aryankamani/stare-dataset-20images/versions/1",
    "FIVES": (
        "nikitamanaenkov/"
        "fundus-image-dataset-for-vessel-segmentation/versions/6"
    ),
}

# فعلاً دو مجموعهٔ کوچک‌تر؛ FIVES در این اجرا دانلود نمی‌شود.
selected_datasets = ["CHASE_DB1", "STARE"]

receipt_dir = work / "data_provenance"
receipt_dir.mkdir(parents=True, exist_ok=True)
EXTRA_DATA_ROOTS = {}

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def _dataset_present(name):
    root = cache_root / catalog[name]
    return root.is_dir() and any(root.rglob('*'))

# اگر همهٔ مجموعه‌های انتخاب‌شده از قبل در کش ماندگار موجود باشند،
# دانلود/استخراج/پردازش تکراری انجام نمی‌شود؛ فقط مسیرها بازخوانی می‌شوند.
if all(_dataset_present(n) for n in selected_datasets):
    print('همهٔ مجموعه‌ها از قبل در کش موجودند؛ دانلود و پردازش تکراری انجام نشد.')
    for n in selected_datasets:
        EXTRA_DATA_ROOTS[n] = (cache_root / catalog[n]).resolve()
        print(f'{n}: از قبل موجود است.')
else:
    for name in selected_datasets:
        handle = catalog[name]
        print(f"\nدریافت {name}: {handle}", flush=True)

        # خطای دانلود متوقف‌کننده است؛ دادهٔ جایگزین تولید نمی‌شود.
        root = Path(kagglehub.dataset_download(handle)).resolve()

        if not root.is_relative_to(cache_root.resolve()):
            raise RuntimeError(
                f"مسیر دریافت خارج از محل ماندگار مورد انتظار است: {root}"
            )

        downloaded_files = sorted(
            p for p in root.rglob("*") if p.is_file()
        )
        if not downloaded_files:
            raise RuntimeError(f"هیچ فایلی برای {name} دریافت نشد.")

        inventory = []
        folders = defaultdict(list)

        for path in downloaded_files:
            relative = path.relative_to(root)
            folders[str(relative.parent)].append(path.name)
            inventory.append({
                "path": str(relative),
                "bytes": path.stat().st_size,
                "sha256": file_sha256(path),
            })

        now = datetime.now(timezone.utc)
        receipt = {
            "dataset": name,
            "handle": handle,
            "kaggle_url": f"https://www.kaggle.com/datasets/{handle}",
            "local_root": str(root),
            "checked_at_utc": now.isoformat(),
            "kagglehub_version": version("kagglehub"),
            "status": "downloaded_pending_annotation_validation",
            "files": inventory,
        }

        receipt_path = receipt_dir / (
            f"{name}_{now.strftime('%Y%m%dT%H%M%S%fZ')}.json"
        )
        with receipt_path.open("x", encoding="utf-8") as stream:
            json.dump(receipt, stream, ensure_ascii=False, indent=2)

        EXTRA_DATA_ROOTS[name] = root
        print("مسیر:", root)
        print("تعداد کل فایل‌ها:", len(downloaded_files))
        print("رسید:", receipt_path)

        for folder, names in sorted(folders.items())[:15]:
            print(f"  {folder}: {len(names)} فایل")
            print("    نمونه:", names[:3])

print("\nدانلود پایان یافت؛ اعتبار برچسب‌ها هنوز بررسی نشده است.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from importlib.metadata import version
import os
import json
import hashlib

if "WORK" not in globals():
    raise RuntimeError("ابتدا سلول ۳، تنظیمات مسیرها، را اجرا کنید.")

if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("ابتدا Google Drive را متصل کنید.")

work = Path(WORK)
cache_root = work / "data_sources" / "_kaggle_cache"
os.environ["KAGGLEHUB_CACHE"] = str(cache_root)
os.environ["DISABLE_COLAB_CACHE"] = "true"

import kagglehub
from kagglehub import config as kagglehub_config

print(
    "کش اختصاصی کولب غیرفعال است؟",
    kagglehub_config.is_colab_cache_disabled()
)
print("مسیر ماندگار دانلود:", cache_root)

catalog = {
    "CHASE_DB1": "namnguynnnn/chase-db1/versions/1",
    "STARE": "aryankamani/stare-dataset-20images/versions/1",
    "FIVES": (
        "nikitamanaenkov/"
        "fundus-image-dataset-for-vessel-segmentation/versions/6"
    ),
}

# فعلاً دو مجموعهٔ کوچک‌تر؛ FIVES در این اجرا دانلود نمی‌شود.
selected_datasets = ["CHASE_DB1", "STARE"]

receipt_dir = work / "data_provenance"
receipt_dir.mkdir(parents=True, exist_ok=True)
EXTRA_DATA_ROOTS = {}

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def _dataset_present(name):
    root = cache_root / catalog[name]
    return root.is_dir() and any(root.rglob('*'))

# اگر همهٔ مجموعه‌های انتخاب‌شده از قبل در کش ماندگار موجود باشند،
# دانلود/استخراج/پردازش تکراری انجام نمی‌شود؛ فقط مسیرها بازخوانی می‌شوند.
if all(_dataset_present(n) for n in selected_datasets):
    print('همهٔ مجموعه‌ها از قبل در کش موجودند؛ دانلود و پردازش تکراری انجام نشد.')
    for n in selected_datasets:
        EXTRA_DATA_ROOTS[n] = (cache_root / catalog[n]).resolve()
        print(f'{n}: از قبل موجود است.')
else:
    for name in selected_datasets:
        handle = catalog[name]
        print(f"\nدریافت {name}: {handle}", flush=True)

        # خطای دانلود متوقف‌کننده است؛ دادهٔ جایگزین تولید نمی‌شود.
        root = Path(kagglehub.dataset_download(handle)).resolve()

        if not root.is_relative_to(cache_root.resolve()):
            raise RuntimeError(
                f"مسیر دریافت خارج از محل ماندگار مورد انتظار است: {root}"
            )

        downloaded_files = sorted(
            p for p in root.rglob("*") if p.is_file()
        )
        if not downloaded_files:
            raise RuntimeError(f"هیچ فایلی برای {name} دریافت نشد.")

        inventory = []
        folders = defaultdict(list)

        for path in downloaded_files:
            relative = path.relative_to(root)
            folders[str(relative.parent)].append(path.name)
            inventory.append({
                "path": str(relative),
                "bytes": path.stat().st_size,
                "sha256": file_sha256(path),
            })

        now = datetime.now(timezone.utc)
        receipt = {
            "dataset": name,
            "handle": handle,
            "kaggle_url": f"https://www.kaggle.com/datasets/{handle}",
            "local_root": str(root),
            "checked_at_utc": now.isoformat(),
            "kagglehub_version": version("kagglehub"),
            "status": "downloaded_pending_annotation_validation",
            "files": inventory,
        }

        receipt_path = receipt_dir / (
            f"{name}_{now.strftime('%Y%m%dT%H%M%S%fZ')}.json"
        )
        with receipt_path.open("x", encoding="utf-8") as stream:
            json.dump(receipt, stream, ensure_ascii=False, indent=2)

        EXTRA_DATA_ROOTS[name] = root
        print("مسیر:", root)
        print("تعداد کل فایل‌ها:", len(downloaded_files))
        print("رسید:", receipt_path)

        for folder, names in sorted(folders.items())[:15]:
            print(f"  {folder}: {len(names)} فایل")
            print("    نمونه:", names[:3])

print("\nدانلود پایان یافت؛ اعتبار برچسب‌ها هنوز بررسی نشده است.")

In [ ]:
from pathlib import Path
from collections import Counter
from PIL import Image

if "WORK" not in globals():
    raise RuntimeError("ابتدا سلول تنظیمات مسیرها را اجرا کنید.")

fives_root = (
    Path(WORK)
    / "data_sources/_kaggle_cache/datasets/nikitamanaenkov"
    / "fundus-image-dataset-for-vessel-segmentation/versions/6"
)

# این سلول چیزی دانلود نمی‌کند؛ اگر FIVES در کش نباشد با پیام روشن متوقف می‌شود
# و هیچ داده‌ای حذف یا دوباره دانلود نمی‌شود.
if not fives_root.is_dir():
    raise FileNotFoundError(
        'FIVES هنوز در کش kagglehub موجود نیست؛ ابتدا سلول دانلود kagglehub را اجرا کنید. '
        'هیچ داده‌ای حذف یا دوباره دانلود نمی‌شود.'
    )

for split, expected in (("train", 600), ("test", 200)):
    image_dir = fives_root / split / "Original"
    mask_dir = fives_root / split / "Ground truth"

    if not image_dir.is_dir() or not mask_dir.is_dir():
        raise FileNotFoundError(f"پوشه‌های مورد انتظار موجود نیستند: {split}")

    print(f"\n--- FIVES / {split} ---", flush=True)

    file_maps = []
    for folder in (image_dir, mask_dir):
        files = sorted(p for p in folder.iterdir() if p.is_file())
        png_files = {p.name: p for p in files if p.suffix.lower() == ".png"}
        other_files = [
            (p.name, p.stat().st_size)
            for p in files if p.suffix.lower() != ".png"
        ]
        print(folder.name, "تعداد PNG:", len(png_files))
        print("فایل‌های خارج از الگوی PNG، نام و بایت:", other_files)
        file_maps.append(png_files)

    images, masks = file_maps
    missing_masks = sorted(set(images) - set(masks))
    missing_images = sorted(set(masks) - set(images))

    print("تصویر بدون ماسک هم‌نام:", missing_masks)
    print("ماسک بدون تصویر هم‌نام:", missing_images)

    if missing_masks or missing_images:
        raise ValueError("تطابق یک‌به‌یک برقرار نیست؛ هیچ فایلی حذف نشد.")
    if len(images) != expected:
        raise ValueError(
            f"تعداد جفت‌های PNG برابر {len(images)} است؛ انتظار: {expected}"
        )

    dimensions = Counter()
    modes = Counter()
    mask_values = set()
    many_color_masks = []

    for number, filename in enumerate(sorted(images), start=1):
        with Image.open(images[filename]) as image, Image.open(masks[filename]) as mask:
            image.load()
            mask.load()

            if image.size != mask.size:
                raise ValueError(
                    f"اختلاف ابعاد در {split}/{filename}: "
                    f"{image.size} در برابر {mask.size}"
                )

            dimensions[image.size] += 1
            modes[(image.mode, mask.mode)] += 1

            # برای ماسک پالتی، رنگ واقعی بررسی می‌شود؛ فایل تغییر نمی‌کند.
            inspected_mask = mask.convert("RGB") if mask.mode == "P" else mask
            colors = inspected_mask.getcolors(maxcolors=256)

            if colors is None:
                many_color_masks.append(filename)
            else:
                mask_values.update(repr(value) for _, value in colors)

        if number % 100 == 0:
            print(f"خوانده شد: {number}/{expected}", flush=True)

    print("جفت‌های PNG خوانا با نام و ابعاد منطبق:", len(images))
    print("ابعاد (عرض، ارتفاع) و تعداد:", dict(dimensions))
    print("حالت رنگی (تصویر، ماسک):", dict(modes))
    print("مقادیر/رنگ‌های ماسک، حداکثر ۲۰ مورد:", sorted(mask_values)[:20])
    print("تعداد ماسک‌های دارای بیش از ۲۵۶ رنگ:", len(many_color_masks))
    print("نمونهٔ نام این ماسک‌ها:", many_color_masks[:10])

print("\nبررسی پایان یافت؛ فایل‌های خارج از الگوی PNG هنوز باید تفسیر شوند.")

In [10]:
import inspect
import scripts.data as data_module
import scripts.training as training_module

for module in (data_module, training_module):
    print("\n" + "=" * 80)
    print("MODULE:", module.__name__)
    print("=" * 80)
    print(inspect.getsource(module))


MODULE: scripts.data
"""Explicit, subject-split manifests. Never synthesize missing research data."""
import json
from pathlib import Path
import numpy as np
from PIL import Image
from .artifacts import sha256_file

DATASETS = ('DRIVE', 'RITE', 'BraTS2020', 'COVID19_CXR')
REQUIRED = ('id', 'dataset', 'patient_id', 'group_id', 'split', 'image', 'mask', 'source', 'mask_definition')

def load_manifest(path, verify_files=True):
    path = Path(path).resolve()
    records = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    if not records: raise ValueError('Empty manifest; real image/mask files required')
    ids, groups, images, file_hashes = set(), {}, {}, {}
    for r in records:
        if any(not r.get(k) for k in REQUIRED): raise ValueError(f'Missing required fields: {REQUIRED}')
        if r['dataset'] not in DATASETS or r['split'] not in ('train', 'val', 'test'):
            raise ValueError('Unknown dataset/split')
        uid = (r['da

In [ ]:
from pathlib import Path
from collections import Counter
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

if "WORK" not in globals():
    raise RuntimeError("ابتدا سلول تنظیمات مسیرها را اجرا کنید.")

cache = Path(WORK) / "data_sources" / "_kaggle_cache" / "datasets"

specs = [
    (
        "CHASE_DB1",
        "namnguynnnn/chase-db1",
        "images", "*.jpg",
        "masks", "_1stHO.png", 28,
    ),
    (
        "STARE",
        "aryankamani/stare-dataset-20images",
        "stare-images", "*.ppm",
        "labels-ah", ".ah.ppm", 20,
    ),
]

for name, handle, image_dir, pattern, mask_dir, suffix, expected in specs:
    root = cache / handle / "versions" / "1"
    print(f"\n--- {name} ---", flush=True)

    if not root.is_dir():
        print(f'{name} در کش موجود نیست؛ ابتدا سلول دانلود kagglehub را اجرا کنید. از این مجموعه عبور می‌شود.')
        continue

    images = sorted((root / image_dir).glob(pattern))
    masks = sorted((root / mask_dir).glob(f"*{suffix}"))

    if len(images) != expected or len(masks) != expected:
        raise ValueError(
            f"{name}: تعداد غیرمنتظره؛ "
            f"تصاویر={len(images)}، ماسک‌ها={len(masks)}، "
            f"انتظار برای هرکدام={expected}"
        )

    expected_masks = {
        root / mask_dir / f"{image.stem}{suffix}"
        for image in images
    }
    if expected_masks != set(masks):
        raise ValueError(
            f"{name}: نام تصاویر و ماسک‌ها یک‌به‌یک تطابق ندارند."
        )

    dimensions = Counter()
    modes = Counter()
    mask_values = set()
    example = None

    for image_path in images:
        mask_path = root / mask_dir / f"{image_path.stem}{suffix}"

        with Image.open(image_path) as image, Image.open(mask_path) as mask:
            image.load()
            mask.load()

            if image.size != mask.size:
                raise ValueError(
                    f"عدم تطابق ابعاد: {image_path.name} "
                    f"{image.size} در برابر {mask.size}"
                )

            dimensions[image.size] += 1
            modes[(image.mode, mask.mode)] += 1
            mask_values.update(np.unique(np.asarray(mask)).tolist())

            if example is None:
                example = (
                    image_path.name,
                    np.asarray(image.convert("RGB")).copy(),
                    np.asarray(mask.convert("RGB")).copy(),
                )

    print("جفت‌های خوانده‌شده با نام و ابعاد منطبق:", len(images))
    print("ابعاد (عرض، ارتفاع) و تعداد:", dict(dimensions))
    print("حالت رنگی (تصویر، ماسک) و تعداد:", dict(modes))
    print("تعداد مقادیر متمایز ذخیره‌شده در ماسک:", len(mask_values))
    print("حداکثر ۲۰ مقدار اول:", sorted(mask_values)[:20])

    filename, image_array, mask_array = example
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(image_array)
    axes[0].set_title(f"{name}: {filename}")
    axes[1].imshow(mask_array)
    axes[1].set_title("Downloaded reference mask")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("\nبررسی ساختاری پایان یافت؛ هنوز تقسیم داده یا آموزش انجام نشده است.")

این کد فقط وضعیت مسیرها را نمایش می‌دهد؛ چیزی دانلود، ایجاد یا تغییر نمی‌دهد:

In [12]:
from pathlib import Path

required = ["WORK", "DATASET", "DATA_ROOT", "SOURCE", "MANIFEST"]
missing = [name for name in required if name not in globals()]

if missing:
    print("ابتدا سلول تنظیمات، یعنی بخش ۳، را اجرا کنید.")
    print("متغیرهای تعریف‌نشده:", missing)
else:
    root = Path(DATA_ROOT)
    manifest = Path(MANIFEST)

    print("DATASET:", DATASET)
    print("WORK:", WORK)
    print("DATA_ROOT:", root)
    print("پوشهٔ داده موجود است؟", root.is_dir())
    print("SOURCE:", repr(SOURCE))
    print("MANIFEST:", manifest)
    print("فایل فهرست موجود است؟", manifest.is_file())

    if root.is_dir():
        entries = sorted(root.iterdir(), key=lambda p: p.name)
        print("\nتعداد موارد مستقیم داخل پوشه:", len(entries))
        print("حداکثر ۲۰ مورد اول:")
        for item in entries[:20]:
            kind = "پوشه" if item.is_dir() else "فایل"
            print(f"  [{kind}] {item.name}")
    else:
        parent = root.parent
        print("\nپوشهٔ والد موجود است؟", parent.is_dir())
        if parent.is_dir():
            print("پوشه‌های موجود در والد:")
            for item in sorted(parent.iterdir(), key=lambda p: p.name):
                if item.is_dir():
                    print(" ", item.name)

DATASET: DRIVE
WORK: /content/drive/MyDrive/Thesis_Research
DATA_ROOT: /content/drive/MyDrive/Thesis_Research/data_sources/DRIVE
پوشهٔ داده موجود است؟ True
SOURCE: ''
MANIFEST: /content/drive/MyDrive/Thesis_Research/DRIVE.jsonl
فایل فهرست موجود است؟ False

تعداد موارد مستقیم داخل پوشه: 2
حداکثر ۲۰ مورد اول:
  [پوشه] test
  [پوشه] training


In [13]:
from pathlib import Path

root = Path(DATA_ROOT)

for split in ("training", "test"):
    split_dir = root / split
    print(f"\n--- {split} ---")

    if not split_dir.is_dir():
        print("پوشه موجود نیست:", split_dir)
        continue

    folders = sorted(
        (p for p in split_dir.iterdir() if p.is_dir()),
        key=lambda p: p.name
    )

    if not folders:
        print("هیچ زیرپوشه‌ای پیدا نشد.")

    for folder in folders:
        items = sorted(
            (p for p in folder.iterdir()
             if p.is_file() and not p.name.startswith(".")),
            key=lambda p: p.name
        )
        print(f"{folder.name}: {len(items)} فایل")
        print("  نمونه نام‌ها:", [p.name for p in items[:3]])


--- training ---
1st_manual: 20 فایل
  نمونه نام‌ها: ['21_manual1.gif', '22_manual1.gif', '23_manual1.gif']
images: 20 فایل
  نمونه نام‌ها: ['21_training.tif', '22_training.tif', '23_training.tif']
mask: 20 فایل
  نمونه نام‌ها: ['21_training_mask.gif', '22_training_mask.gif', '23_training_mask.gif']

--- test ---
images: 20 فایل
  نمونه نام‌ها: ['01_test.tif', '02_test.tif', '03_test.tif']
mask: 20 فایل
  نمونه نام‌ها: ['01_test_mask.gif', '02_test_mask.gif', '03_test_mask.gif']


In [14]:
import inspect
from scripts.prepare import retinal
from scripts.data import load_manifest

for function in (retinal, load_manifest):
    print("\n" + "=" * 70)
    print(f"FUNCTION: {function.__module__}.{function.__name__}")
    print("=" * 70)
    print(inspect.getsource(function))


FUNCTION: scripts.prepare.retinal
def retinal(root, dataset, source):
    records = []
    for official in ('training', 'test'):
        folder = root / official
        images = sorted((folder / 'images').glob('*'))
        images = [p for p in images if p.suffix.lower() in ('.tif', '.tiff', '.png', '.jpg')]
        for image in images:
            ident = image.stem.split('_')[0]
            if not ident.isdigit(): raise ValueError(f'Cannot infer original DRIVE/RITE ID: {image}')
            ident = f'{int(ident):02d}'
            masks = [p for p in folder.rglob('*') if p.is_file() and p.stem.split('_')[0] == ident
                     and (('1st_manual' in str(p)) if dataset == 'DRIVE' else ('av' in p.parent.name.lower()))]
            if len(masks) != 1: raise ValueError(f'Need exactly one first-observer/AV mask for {image}: {masks}')
            # Official test untouched; same four train IDs held out in both retinal datasets.
            split = 'test' if official == 'test' else

In [19]:
import torch

print('torch:', torch.__version__)
print('torch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())

DEVICE = 'cpu'
if torch.cuda.is_available():
    try:
        probe = torch.ones(4, device='cuda')
        print('CUDA probe:', (probe * 2).sum().item())
        print('GPU:', torch.cuda.get_device_name(0))
        del probe
        DEVICE = 'cuda'
    except RuntimeError as exc:
        print('CUDA initialization failed; using CPU:', repr(exc))

probe = torch.ones(4, device=DEVICE)
print('Selected device:', DEVICE)
print('Device test:', (probe * 2).sum().item())
del probe

torch: 2.11.0+cpu
torch CUDA build: None
CUDA available: False
Selected device: cpu
Device test: 8.0


## ۴. ساخت فهرست دادهٔ واقعی
اگر ساختار شما متفاوت است، مطابق README فایل JSONL صریح تهیه کنید. این سلول هیچ داده‌ای دانلود یا جعل نمی‌کند. برای BraTS اسلایس‌ها بعد از تقسیم بیمار ساخته می‌شوند؛ DRIVE/RITE از گروه‌های مشترک استفاده می‌کنند.

In [15]:
import hashlib
import json
import tempfile
from pathlib import Path
from collections import Counter
from PIL import Image
from scripts.data import load_manifest, load_sample

RUN_TRAINING = False
RUN_EXPERIMENTS = False
METHOD_READY = False


def build_drive_baseline(work, data_root, load_manifest_fn, load_sample_fn):
    work, root = Path(work).resolve(), Path(data_root).resolve()

    receipt = json.loads(
        (work / "drive_download_report.json").read_text(encoding="utf-8")
    )

    if (
        receipt.get("status") != "download_verified"
        or Path(receipt.get("data_root", "")).resolve() != root
    ):
        raise ValueError("Verified DRIVE receipt and matching data root required")

    if any(
        not isinstance(receipt.get(k), str) or not receipt[k].strip()
        for k in ("source_page", "distribution", "sha256")
    ):
        raise ValueError("Missing source metadata in DRIVE receipt")

    source = (
        f"{receipt['source_page']} | {receipt['distribution']} | "
        f"archive_sha256={receipt['sha256']}"
    )
    val_ids = {"21", "26", "31", "36"}

    def check_images(folder, suffix, expected):
        found = sorted(p for p in folder.glob("*.tif") if p.is_file())
        wanted = {f"{i:02d}_{suffix}.tif" for i in expected}
        if {p.name for p in found} != wanted:
            raise ValueError(f"Unexpected DRIVE image list: {folder}")
        return found

    rows, inference = [], []

    for image in check_images(
        root / "training/images", "training", range(21, 41)
    ):
        ident = image.stem.split("_")[0]
        mask = root / "training/1st_manual" / f"{ident}_manual1.gif"
        rows.append(dict(
            id=ident,
            dataset="DRIVE",
            patient_id=ident,
            group_id=f"retina:{ident}",
            split="val" if ident in val_ids else "train",
            image=str(image),
            mask=str(mask),
            source=source,
            mask_definition="first-observer vessels",
            split_protocol="DRIVE-baseline-v1",
        ))

    for image in check_images(
        root / "test/images", "test", range(1, 21)
    ):
        ident = image.stem.split("_")[0]
        inference.append(dict(
            id=ident,
            dataset="DRIVE",
            patient_id=ident,
            group_id=f"retina:{ident}",
            split="test",
            image=str(image),
            mask=None,
            source=source,
            has_ground_truth=False,
            purpose="prediction only; no local test Dice/IoU",
        ))

    # اعتبارسنجی با همان بارگذار واقعی برنامه و محاسبهٔ هش فایل‌ها.
    with tempfile.TemporaryDirectory(prefix="drive-check-") as temporary:
        stage = Path(temporary) / "input.jsonl"
        stage.write_text(
            "\n".join(json.dumps(r) for r in rows) + "\n",
            encoding="utf-8",
        )
        rows = load_manifest_fn(stage)

    # بررسی هندسهٔ ماسک و تکرار دقیق تصویرِ خاکستری میان تمام ۴۰ نمونه.
    decoded = {}

    for record in rows + inference:
        if record.get("mask") is not None:
            image_array, mask_array, _ = load_sample_fn(record)
            if not mask_array.any() or mask_array.all():
                raise ValueError(f"Empty/full vessel mask: {record['id']}")
            height, width = image_array.shape
            pixels = image_array.tobytes()
        else:
            with Image.open(record["image"]) as im:
                im.load()
                luminance = im.convert("L")
                width, height = luminance.size
                pixels = luminance.tobytes()

            record["image_sha256"] = hashlib.sha256(
                Path(record["image"]).read_bytes()
            ).hexdigest()

        digest = hashlib.sha256(
            f"{width}x{height}:".encode() + pixels
        ).hexdigest()

        if digest in decoded:
            raise ValueError(
                f"Duplicate decoded DRIVE image: "
                f"{decoded[digest]} and {record['id']}"
            )

        decoded[digest] = record["id"]
        record["image_luminance_sha256"] = digest

    def jsonl(items):
        return "".join(
            json.dumps(r, sort_keys=True, ensure_ascii=False) + "\n"
            for r in items
        )

    out = work / "manifests"
    trainval = out / "DRIVE_train_val_v1.jsonl"
    test = out / "DRIVE_test_unlabeled_v1.jsonl"

    payloads = {
        trainval: jsonl(rows),
        test: jsonl(inference),
    }

    protocol = {
        "protocol": "DRIVE-baseline-v1",
        "validation_ids": sorted(val_ids),
        "train_count": 16,
        "val_count": 4,
        "unlabeled_test_count": 20,
        "source": source,
        "test_used_for_training": False,
        "duplicate_check": (
            "exact decoded Pillow L pixels; "
            "does not detect all near duplicates"
        ),
        "trainval_sha256": hashlib.sha256(
            payloads[trainval].encode()
        ).hexdigest(),
        "unlabeled_test_sha256": hashlib.sha256(
            payloads[test].encode()
        ).hexdigest(),
    }

    payloads[out / "DRIVE_baseline_protocol_v1.json"] = (
        json.dumps(
            protocol, ensure_ascii=False, sort_keys=True, indent=2
        ) + "\n"
    )

    # فایل متفاوتِ قبلی بازنویسی نمی‌شود.
    for path, content in payloads.items():
        if path.exists() and path.read_text(encoding="utf-8") != content:
            raise FileExistsError(
                f"Existing manifest differs; not overwritten: {path}"
            )

    out.mkdir(parents=True, exist_ok=True)

    with tempfile.TemporaryDirectory(
        prefix=".drive-publish-", dir=out
    ) as temporary:
        for path, content in payloads.items():
            staged = Path(temporary) / path.name
            staged.write_text(content, encoding="utf-8")

        for path in payloads:
            if not path.exists():
                (Path(temporary) / path.name).replace(path)

    return trainval, test, rows, source


if DATASET != "DRIVE":
    raise ValueError("این سلول برای آزمایش پایهٔ DRIVE است.")

MANIFEST, UNLABELED_TEST_MANIFEST, records, SOURCE = build_drive_baseline(
    WORK, DATA_ROOT, load_manifest, load_sample
)

WEIGHTS = Path(WORK) / "weights" / "DRIVE_baseline_v1"

print("تقسیم داده:", dict(Counter(r["split"] for r in records)))
print("فهرست آموزش/اعتبارسنجی:", MANIFEST)
print("فهرست جداگانهٔ آزمون بدون برچسب:", UNLABELED_TEST_MANIFEST)
print("منشأ ثبت‌شده:", SOURCE)
print("مرحلهٔ ۴ پایان یافت؛ هیچ آموزشی انجام نشد.")

تقسیم داده: {'val': 4, 'train': 16}
فهرست آموزش/اعتبارسنجی: /content/drive/MyDrive/Thesis_Research/manifests/DRIVE_train_val_v1.jsonl
فهرست جداگانهٔ آزمون بدون برچسب: /content/drive/MyDrive/Thesis_Research/manifests/DRIVE_test_unlabeled_v1.jsonl
منشأ ثبت‌شده: https://www.kaggle.com/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction | Kaggle redistribution; version 1 | archive_sha256=3efa1bf264da71a9080c9959c5ee89e2646193892efe9d66f29a4123b74f949a
مرحلهٔ ۴ پایان یافت؛ هیچ آموزشی انجام نشد.


In [16]:
import torch
import json
from pathlib import Path

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

profile = json.loads(Path(PROFILE).read_text(encoding="utf-8"))
print("\nTRAINING_CONFIG:")
print(json.dumps(profile["training"], ensure_ascii=False, indent=2))

print("\nINSTALL_COLAB_SOURCE:")
print(
    (Path(PACKAGE) / "scripts/install_colab.py").read_text(encoding="utf-8")
)

PyTorch: 2.11.0+cpu
CUDA build: None
GPU available: False

TRAINING_CONFIG:
{
  "epochs": 100,
  "batch_size": 4,
  "learning_rate": 0.001,
  "base_channels": 32,
  "size": 256,
  "patience": 15,
  "threshold": 0.5
}

INSTALL_COLAB_SOURCE:
"""Install missing dependencies without replacing an existing CUDA-enabled torch."""
import subprocess
import sys
from pathlib import Path

PACKAGES = ['numpy', 'scipy', 'scikit-image', 'pillow', 'numba', 'nibabel',
            'cryptography', 'torch', 'matplotlib', 'pytest', 'nbformat']

if __name__ == '__main__':
    subprocess.run([sys.executable, '-m', 'pip', 'install', *PACKAGES], check=True)
    result = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True, capture_output=True, text=True)
    Path('installed_environment.txt').write_text(result.stdout, encoding='utf-8')
    print('نصب انجام شد. نسخه‌ها در installed_environment.txt ثبت شدند؛ CUDA موجود با نسخهٔ CPU جایگزین نشد.')


In [17]:
import shutil
import subprocess
import torch

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("PyTorch GPU available:", torch.cuda.is_available())

nvidia_smi = shutil.which("nvidia-smi")

if nvidia_smi is None:
    print("\nNVIDIA_STATUS: command_not_found")
    print("ابزار شناسایی GPU انویدیا در این نشست پیدا نشد.")
else:
    result = subprocess.run(
        [
            nvidia_smi,
            "--query-gpu=name,driver_version,memory.total",
            "--format=csv,noheader",
        ],
        capture_output=True,
        text=True,
        timeout=20,
    )
    print("\nNVIDIA_STATUS: return_code =", result.returncode)
    print("GPU / Driver / Memory:")
    print(result.stdout.strip() or "(خروجی خالی)")
    if result.stderr.strip():
        print("پیام ابزار:", result.stderr.strip())

PyTorch: 2.11.0+cpu
CUDA build: None
PyTorch GPU available: False

NVIDIA_STATUS: command_not_found
ابزار شناسایی GPU انویدیا در این نشست پیدا نشد.


cpu verion

In [20]:
from pathlib import Path
from collections import Counter
import numpy as np
from scripts.data import load_manifest, load_sample

required = ['MANIFEST', 'DATASET', 'PROFILE', 'WEIGHTS', 'DEVICE']
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f'Run setup/config cells first; missing: {missing}')

if not Path(MANIFEST).is_file():
    raise FileNotFoundError('Manifest missing: section 4 must succeed first')

records = load_manifest(MANIFEST)
selected = [r for r in records if r.get('dataset') == DATASET]
counts = Counter(r.get('split') for r in selected)
print('Dataset:', DATASET)
print('Split counts:', dict(counts))
print('RUN_TRAINING:', globals().get('RUN_TRAINING'))
print('RESUME_TRAINING:', globals().get('RESUME_TRAINING'))
print('best.pt exists:', (Path(WEIGHTS) / 'best.pt').is_file())
if not selected:
    raise ValueError('No records for selected DATASET')

def summary(a):
    a = np.asarray(a)
    if a.size == 0:
        raise ValueError('Empty image or mask')
    return {
        'shape': list(a.shape), 'dtype': str(a.dtype),
        'finite': bool(np.isfinite(a).all()),
        'min': float(a.min()), 'max': float(a.max()),
        'nonzero_fraction': float(np.count_nonzero(a) / a.size),
    }

# A small diagnostic sample, not a full dataset audit.
for split in ('train', 'val', 'valid', 'validation'):
    subset = [r for r in selected if r.get('split') == split]
    for i, record in enumerate(subset[:2]):
        image, mask, _ = load_sample(record)
        print(f'\n{split} sample {i}')
        print('image:', summary(image))
        print('mask:', summary(mask))
        values = np.unique(np.asarray(mask))
        print('mask unique count:', len(values))
        print('mask first unique values:', values[:16].tolist())

Dataset: DRIVE
Split counts: {'val': 4, 'train': 16}
RUN_TRAINING: False
RESUME_TRAINING: False
best.pt exists: False

train sample 0
image: {'shape': [584, 565], 'dtype': 'uint8', 'finite': True, 'min': 0.0, 'max': 217.0, 'nonzero_fraction': 0.9848254333858649}
mask: {'shape': [584, 565], 'dtype': 'bool', 'finite': True, 'min': 0.0, 'max': 1.0, 'nonzero_fraction': 0.09034125348527094}
mask unique count: 2
mask first unique values: [False, True]

train sample 1
image: {'shape': [584, 565], 'dtype': 'uint8', 'finite': True, 'min': 0.0, 'max': 244.0, 'nonzero_fraction': 0.9848890774639351}
mask: {'shape': [584, 565], 'dtype': 'bool', 'finite': True, 'min': 0.0, 'max': 1.0, 'nonzero_fraction': 0.06583525275791005}
mask unique count: 2
mask first unique values: [False, True]

val sample 0
image: {'shape': [584, 565], 'dtype': 'uint8', 'finite': True, 'min': 0.0, 'max': 243.0, 'nonzero_fraction': 0.9851375924354467}
mask: {'shape': [584, 565], 'dtype': 'bool', 'finite': True, 'min': 0.0, 'm

## ۵. آموزش و ارزیابی U-Net
آموزش به عددهای ODE نیاز ندارد. `RUN_TRAINING=True` را پس از بررسی داده فعال کنید؛ دوره‌ها و دیگر تنظیمات در کپی JSON قابل تغییرند. بهترین وزن فقط از اعتبارسنجی انتخاب می‌شود. با قطع نشست و همان تنظیمات، RESUME_TRAINING را فعال کنید.

cpu version

In [23]:
from pathlib import Path
import copy, json, uuid
from google.colab import files
from scripts.training import train

cfg_debug = copy.deepcopy(
    json.loads(Path(PROFILE).read_text(encoding='utf-8'))
)
cfg_debug['training']['epochs'] = 3

DEBUG_WEIGHTS = (
    Path(WORK) / 'debug_training'
    / f'{DATASET}_{uuid.uuid4().hex[:8]}'
)
# Do not pre-populate this directory: train() manages its creation.
print('Device:', DEVICE, flush=True)
print('Diagnostic output:', DEBUG_WEIGHTS, flush=True)
print('Starting up to 3 epochs; CPU training may take time.', flush=True)

try:
    result = train(
        MANIFEST, DATASET, cfg_debug, DEBUG_WEIGHTS,
        device=DEVICE, resume=False
    )
    print('Result:', result, flush=True)
finally:
    history_file = DEBUG_WEIGHTS / 'history.json'
    if history_file.is_file():
        print(history_file.read_text(encoding='utf-8'))
        files.download(str(history_file))
    else:
        print('No completed epoch history was saved; send any traceback.')


Device: cpu
Diagnostic output: /content/drive/MyDrive/Thesis_Research/debug_training/DRIVE_07e4d848
Starting up to 3 epochs; CPU training may take time.
{'epoch': 1, 'train_loss': 1.598048985004425, 'train_dice': 0.16084483689264595, 'val_loss': 1.5762927532196045, 'val_dice': 0.1509948090918858}
{'epoch': 2, 'train_loss': 1.698235034942627, 'train_dice': 0.04102360374825099, 'val_loss': 1.3239021301269531, 'val_dice': 0.0}
{'epoch': 3, 'train_loss': 1.424532264471054, 'train_dice': 0.0, 'val_loss': 1.4841490983963013, 'val_dice': 0.0}
Result: {'best_checkpoint': '/content/drive/MyDrive/Thesis_Research/debug_training/DRIVE_07e4d848/best.pt', 'epochs': 3, 'best_val_dice': 0.1509948090918858}
[
  {
    "epoch": 1,
    "train_loss": 1.598048985004425,
    "train_dice": 0.16084483689264595,
    "val_loss": 1.5762927532196045,
    "val_dice": 0.1509948090918858
  },
  {
    "epoch": 2,
    "train_loss": 1.698235034942627,
    "train_dice": 0.04102360374825099,
    "val_loss": 1.323902130126

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

gpu
version

In [18]:
import os
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone
import torch

RUN_TRAINING = False
RUN_EXPERIMENTS = False
METHOD_READY = False
SMOKE_READY = False

required = ["WORK", "PACKAGE", "MANIFEST", "PROFILE"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"ابتدا تنظیمات و مرحلهٔ ۴ را اجرا کنید: {missing}")

if not torch.cuda.is_available():
    raise RuntimeError("GPU در دسترس PyTorch نیست؛ آموزش با CPU جایگزین نشد.")

# فقط یک کپی از تنظیمات؛ فایل PROFILE تغییر نمی‌کند.
smoke_config = json.loads(Path(PROFILE).read_text(encoding="utf-8"))
smoke_config["training"]["epochs"] = 2

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
SMOKE_DIR = Path(WORK) / "checks" / f"DRIVE_smoke_{stamp}"
SMOKE_DIR.mkdir(parents=True, exist_ok=False)

config_file = SMOKE_DIR / "smoke_config.json"
config_file.write_text(
    json.dumps(smoke_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# اجرا در پردازشی تازه، با تنظیمات CUDA پیش از واردکردن PyTorch.
worker_code = r'''
import hashlib
import json
import sys
from collections import Counter
from pathlib import Path
import torch
from scripts.data import load_manifest, load_sample
from scripts.training import train
from scripts.model import UNet

manifest, config_file, out = map(Path, sys.argv[1:4])
device = sys.argv[4]

if device == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable; no CPU fallback')

cfg = json.loads(config_file.read_text(encoding='utf-8'))
if cfg['training']['epochs'] != 2:
    raise ValueError('This check requires exactly two epochs')

rows = load_manifest(manifest)
if any(r['dataset'] != 'DRIVE' for r in rows):
    raise ValueError('This check is DRIVE baseline only')
if Counter(r['split'] for r in rows) != Counter({'train':16, 'val':4}):
    raise ValueError('Expected 16 training and 4 validation records, no test records')
if {r['id'] for r in rows if r['split']=='val'} != {'21','26','31','36'}:
    raise ValueError('Validation partition changed')

result = train(
    manifest, 'DRIVE', cfg, out/'weights', device, resume=False
)

best = torch.load(
    out/'weights/best.pt', map_location='cpu', weights_only=True
)
last = torch.load(
    out/'weights/last.pt', map_location='cpu', weights_only=True
)

expected_hash = hashlib.sha256(manifest.read_bytes()).hexdigest()
expected_groups = sorted({r['group_id'] for r in rows})

for checkpoint in (best, last):
    if (
        checkpoint['dataset'] != 'DRIVE'
        or checkpoint['manifest_sha256'] != expected_hash
    ):
        raise ValueError('Checkpoint dataset/manifest mismatch')
    if (
        checkpoint['training'] != cfg['training']
        or checkpoint['seen_groups'] != expected_groups
    ):
        raise ValueError('Checkpoint configuration/group mismatch')

if last['trained_epochs'] != 2:
    raise ValueError('Two epochs did not complete')

# بارگذاری سخت‌گیرانهٔ وزن‌ها در نمونهٔ تازه‌ای از همان معماری.
net = UNet(cfg['training']['base_channels']).to(device)
net.load_state_dict(best['state_dict'], strict=True)
net.eval()

reference = next(r for r in rows if r['split']=='val')
image, target, _ = load_sample(reference, cfg['training']['size'])
x = torch.from_numpy(image).float()[None, None].to(device) / 255.0

with torch.inference_mode():
    logits = net(x)

if (
    tuple(logits.shape) != (1, 1, *target.shape)
    or not torch.isfinite(logits).all().item()
):
    raise ValueError('Reloaded model output shape/values invalid')

check = {
    **result,
    'status': 'software_flow_check_passed',
    'device': device,
    'last_trained_epochs': last['trained_epochs'],
    'best_trained_epoch': best['trained_epochs'],
    'checkpoint_reload_verified': True,
    'output_shape': list(logits.shape),
    'test_used': False,
    'purpose': 'two-epoch pipeline check; not final thesis results',
}

(out/'check_result.json').write_text(
    json.dumps(check, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(check, ensure_ascii=False, indent=2), flush=True)
'''

env = os.environ.copy()
env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

command = [
    sys.executable, "-u", "-c", worker_code,
    str(Path(MANIFEST).resolve()),
    str(config_file.resolve()),
    str(SMOKE_DIR.resolve()),
    "cuda",
]

print("GPU:", torch.cuda.get_device_name(0))
print("شروع آموزش دو‌دوره‌ای؛ خروجی‌ها در این پوشه ثبت می‌شوند:")
print(SMOKE_DIR, flush=True)

log_path = SMOKE_DIR / "train.log"

with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        command,
        cwd=str(Path(PACKAGE).resolve()),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    return_code = process.wait()

if return_code != 0:
    (SMOKE_DIR / "failure.json").write_text(
        json.dumps({
            "status": "failed",
            "return_code": return_code,
            "log": str(log_path),
        }, indent=2),
        encoding="utf-8",
    )
    raise RuntimeError(f"اجرای کوتاه متوقف شد؛ گزارش خطا: {log_path}")

SMOKE_RESULT = json.loads(
    (SMOKE_DIR / "check_result.json").read_text(encoding="utf-8")
)
SMOKE_READY = SMOKE_RESULT["status"] == "software_flow_check_passed"

print("\nبررسی اجرای کوتاه موفق بود؟", SMOKE_READY)
print("گزارش:", SMOKE_DIR / "check_result.json")
print("این وزن‌ها مخصوص بررسی مسیر هستند؛ آموزش اصلی هنوز شروع نشده است.")

RuntimeError: GPU در دسترس PyTorch نیست؛ آموزش با CPU جایگزین نشد.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import torch
from scripts.data import load_manifest

# اجرای اصلی آموزش با توقف زودهنگام ملایم (patience=15) روی DRIVE با GPU.
RUN_TRAINING = RUN_EXPERIMENTS = RUN_NIST = RUN_DELIVERY = False
METHOD_READY = MAIN_TRAINING_COMPLETED = False

required = ['WORK', 'PACKAGE', 'MANIFEST', 'PROFILE', 'DATASET']
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f'ابتدا تنظیمات و مرحلهٔ ۴ را اجرا کنید: {missing}')

WORK, PACKAGE, MANIFEST, PROFILE, DATASET = [
    globals()[name] for name in required
]

if DATASET != 'DRIVE' or not torch.cuda.is_available():
    raise RuntimeError('این اجرا به دادهٔ DRIVE و GPU فعال نیاز دارد.')

manifest = Path(MANIFEST).resolve()
expected_manifest = Path(WORK) / 'manifests/DRIVE_train_val_v1.jsonl'
if manifest != expected_manifest.resolve():
    raise ValueError('فهرست انتخاب‌شده، فهرست پایهٔ مرحلهٔ ۴ نیست.')

rows = load_manifest(manifest)
if (
    any(r['dataset'] != 'DRIVE' for r in rows)
    or Counter(r['split'] for r in rows) != Counter(train=16, val=4)
    or {r['id'] for r in rows if r['split'] == 'val'}
       != {'21', '26', '31', '36'}
    or {r['id'] for r in rows} != {str(i) for i in range(21, 41)}
):
    raise ValueError('تقسیم ثابت ۱۶ آموزش و ۴ اعتبارسنجی تغییر کرده است.')

main_cfg = json.loads(Path(PROFILE).read_text(encoding='utf-8'))
expected_training = dict(
    epochs=100, batch_size=4, learning_rate=0.001,
    base_channels=32, size=256, patience=15, threshold=0.5
)
if main_cfg['training'] != expected_training or main_cfg['seed'] != 2026:
    raise ValueError('تنظیمات با خط پایهٔ بررسی‌شده متفاوت است؛ بازنویسی نشد.')

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
MAIN_DIR = Path(WORK) / 'training_runs' / f'DRIVE_baseline_v1_{stamp}'
if MAIN_DIR.exists():
    raise FileExistsError(f'پوشه موجود است؛ حذف یا بازنویسی نکنید: {MAIN_DIR}')

MAIN_DIR.mkdir(parents=True, exist_ok=False)
main_profile = MAIN_DIR / 'config.json'
main_profile.write_text(
    json.dumps(main_cfg, ensure_ascii=False, indent=2),
    encoding='utf-8'
)
MAIN_WEIGHTS = MAIN_DIR / 'weights'
main_log = MAIN_DIR / 'train.log'

# تنظیم CUDA پیش از ورود PyTorch در پردازش تازه؛ بدون ادامهٔ وزن دودوره‌ای.
env = os.environ.copy()
env['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

command = [
    sys.executable, '-u', '-m', 'scripts', 'train',
    '--manifest', str(manifest),
    '--dataset', 'DRIVE',
    '--config', str(main_profile.resolve()),
    '--output', str(MAIN_WEIGHTS.resolve()),
    '--device', 'cuda'
]

print('GPU:', torch.cuda.get_device_name(0))
print('آموزش پایه از صفر؛ حداکثر ۱۰۰ دوره با توقف زودهنگام.', flush=True)
print('پوشهٔ خروجی:', MAIN_DIR, flush=True)

with main_log.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(
        command,
        cwd=str(Path(PACKAGE).resolve()),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise

if return_code != 0:
    raise RuntimeError(f'آموزش متوقف شد؛ خروجی خطا را بفرستید: {main_log}')

MAIN_CHECKPOINT = MAIN_WEIGHTS / 'best.pt'
if not MAIN_CHECKPOINT.is_file() or not (MAIN_WEIGHTS / 'last.pt').is_file():
    raise RuntimeError('فایل‌های وزن پس از اجرا کامل نیستند.')

history = json.loads(
    (MAIN_WEIGHTS / 'history.json').read_text(encoding='utf-8')
)
if not history:
    raise RuntimeError('تاریخچهٔ آموزش خالی است.')

best_epoch = max(
    history,
    key=lambda r: (r['val_dice'], -r['val_loss'])
)

MAIN_TRAINING_COMPLETED = True
print('\nتعداد دوره‌های انجام‌شده:', len(history))
print('بهترین دوره بر مبنای اعتبارسنجی:', best_epoch)
print('وزن منتخب:', MAIN_CHECKPOINT)
print(
    'این نتیجهٔ اعتبارسنجی است، نه Dice آزمون یا تأیید امنیت؛ '
    'پیش‌بررسی بخش ۶ اکنون اجرا می‌شود؛ اگر پارامترهای رابطهٔ ۳-۲ مستند نشده باشند، همان‌جا راه‌های مشروع اعلام می‌شود.'
)

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import torch
from scripts.data import load_manifest

# اجرای اصلی آموزش با توقف زودهنگام ملایم (patience=100) روی DRIVE با GPU.
RUN_TRAINING = RUN_EXPERIMENTS = RUN_NIST = RUN_DELIVERY = False
METHOD_READY = MAIN_TRAINING_COMPLETED = False

required = ['WORK', 'PACKAGE', 'MANIFEST', 'PROFILE', 'DATASET']
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f'ابتدا تنظیمات و مرحلهٔ ۴ را اجرا کنید: {missing}')

WORK, PACKAGE, MANIFEST, PROFILE, DATASET = [
    globals()[name] for name in required
]

if DATASET != 'DRIVE' or not torch.cuda.is_available():
    raise RuntimeError('این اجرا به دادهٔ DRIVE و GPU فعال نیاز دارد.')

manifest = Path(MANIFEST).resolve()
expected_manifest = Path(WORK) / 'manifests/DRIVE_train_val_v1.jsonl'
if manifest != expected_manifest.resolve():
    raise ValueError('فهرست انتخاب‌شده، فهرست پایهٔ مرحلهٔ ۴ نیست.')

rows = load_manifest(manifest)
if (
    any(r['dataset'] != 'DRIVE' for r in rows)
    or Counter(r['split'] for r in rows) != Counter(train=16, val=4)
    or {r['id'] for r in rows if r['split'] == 'val'}
       != {'21', '26', '31', '36'}
    or {r['id'] for r in rows} != {str(i) for i in range(21, 41)}
):
    raise ValueError('تقسیم ثابت ۱۶ آموزش و ۴ اعتبارسنجی تغییر کرده است.')

main_cfg = json.loads(Path(PROFILE).read_text(encoding='utf-8'))
expected_training = dict(
    epochs=100, batch_size=4, learning_rate=0.001,
    base_channels=32, size=256, patience=15, threshold=0.5
)
if main_cfg['training'] != expected_training or main_cfg['seed'] != 2026:
    raise ValueError('تنظیمات با خط پایهٔ بررسی‌شده متفاوت است؛ بازنویسی نشد.')

# تنها تغییر آزمایشی نسبت به خط پایه:
main_cfg['training']['patience'] = 100

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
MAIN_DIR = Path(WORK) / 'training_runs' / f'DRIVE_patience100_v1_{stamp}'
if MAIN_DIR.exists():
    raise FileExistsError(f'پوشه موجود است؛ حذف یا بازنویسی نکنید: {MAIN_DIR}')

MAIN_DIR.mkdir(parents=True, exist_ok=False)
main_profile = MAIN_DIR / 'config.json'
main_profile.write_text(
    json.dumps(main_cfg, ensure_ascii=False, indent=2),
    encoding='utf-8'
)
MAIN_WEIGHTS = MAIN_DIR / 'weights'
main_log = MAIN_DIR / 'train.log'

# تنظیم CUDA پیش از ورود PyTorch در پردازش تازه؛ بدون ادامهٔ وزن دودوره‌ای.
env = os.environ.copy()
env['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

command = [
    sys.executable, '-u', '-m', 'scripts', 'train',
    '--manifest', str(manifest),
    '--dataset', 'DRIVE',
    '--config', str(main_profile.resolve()),
    '--output', str(MAIN_WEIGHTS.resolve()),
    '--device', 'cuda'
]

print('GPU:', torch.cuda.get_device_name(0))
print('آموزش پایه از صفر؛ حداکثر ۱۰۰ دوره با توقف زودهنگام.', flush=True)
print('پوشهٔ خروجی:', MAIN_DIR, flush=True)

with main_log.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(
        command,
        cwd=str(Path(PACKAGE).resolve()),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise

if return_code != 0:
    raise RuntimeError(f'آموزش متوقف شد؛ خروجی خطا را بفرستید: {main_log}')

MAIN_CHECKPOINT = MAIN_WEIGHTS / 'best.pt'
if not MAIN_CHECKPOINT.is_file() or not (MAIN_WEIGHTS / 'last.pt').is_file():
    raise RuntimeError('فایل‌های وزن پس از اجرا کامل نیستند.')

history = json.loads(
    (MAIN_WEIGHTS / 'history.json').read_text(encoding='utf-8')
)
if not history:
    raise RuntimeError('تاریخچهٔ آموزش خالی است.')

best_epoch = max(
    history,
    key=lambda r: (r['val_dice'], -r['val_loss'])
)

MAIN_TRAINING_COMPLETED = True
print('\nتعداد دوره‌های انجام‌شده:', len(history))
print('بهترین دوره بر مبنای اعتبارسنجی:', best_epoch)
print('وزن منتخب:', MAIN_CHECKPOINT)
print(
    'این نتیجهٔ اعتبارسنجی است، نه Dice آزمون یا تأیید امنیت؛ '
    'پیش‌بررسی بخش ۶ اکنون اجرا می‌شود؛ اگر پارامترهای رابطهٔ ۳-۲ مستند نشده باشند، همان‌جا راه‌های مشروع اعلام می‌شود.'
)

## ۶. اعتبار روش پیش از رمزنگاری
این سلول خودکفا است و `cfg`، `CHECKPOINT` و نمونهٔ آزمون را خودش تعریف می‌کند. پارامترهای رابطهٔ ۳-۲ عمداً ناقص‌اند؛ اگر پیش‌بررسی با فهرست «پارامترهای تعیین‌نشده» متوقف شد، دو راه مشروع دارید: (۱) عددها را در پروفایل مستند خودتان تعیین و `PROFILE` را به آن اشاره دهید، یا (۲) `USE_APPENDIX_PROFILE=True` تا پروفایل پیوست (`scripts/appendix_experiment.json`؛ اعداد مستند پیوست پایان‌نامه) اجرا شود و نتیجه با تفسیر «آزمایش پیوست» گزارش شود. پروفایل پیوست جایگزین بی‌نام رابطهٔ ۳-۲ نیست و عددهای مطلوب دلیل انتخاب پارامتر نیستند.


In [ ]:
# ۶. پیش‌بررسی روش رمزنگاری — خودکفا: cfg، CHECKPOINT و نمونهٔ آزمون را همین‌جا تعریف می‌کند.
import json
from pathlib import Path
from scripts.config import validate
from scripts.model import Predictor
from scripts.chaos import roi_digest, stream
from scripts.artifacts import write_json
from scripts.data import load_manifest, load_sample

METHOD_READY = False

# رابطهٔ ۳-۲ در پایان‌نامه عمداً بدون عدد است و پروفایل پیش‌فرض همین را منعکس می‌کند.
# پیش‌فرض: True — اجرا با اعداد مستند پیوست پایان‌نامه (DOCX، بازهٔ P1242-P1249)؛
# نتیجه فقط با عنوان «آزمایش پیوست» قابل گزارش است. برای رابطهٔ ۳-۲ باید False کنید
# و عددها را در پروفایل مستند خودتان تعیین کنید.
USE_APPENDIX_PROFILE = True

if USE_APPENDIX_PROFILE:
    cfg = json.loads((Path(PACKAGE)/'scripts/appendix_experiment.json').read_text(encoding='utf-8'))
else:
    cfg = json.loads(Path(PROFILE).read_text(encoding='utf-8'))

CHECKPOINT = globals().get('MAIN_CHECKPOINT')
if CHECKPOINT is None or not Path(CHECKPOINT).is_file():
    fallback = globals().get('WEIGHTS')
    CHECKPOINT = Path(fallback)/'best.pt' if fallback else None
if CHECKPOINT is None or not Path(CHECKPOINT).is_file():
    raise FileNotFoundError('وزن آموزش‌دیده یافت نشد؛ مرحلهٔ ۵ را کامل کنید یا MAIN_CHECKPOINT را دستی مقدار دهید.')

records = [r for r in load_manifest(MANIFEST) if r['dataset'] == DATASET]
selected = next((r for r in records if r['split'] == 'test'), None) \
           or next((r for r in records if r['split'] == 'val'), None)
if selected is None:
    raise ValueError('نمونهٔ آزمون/اعتبارسنجی در فهرست نیست؛ مرحلهٔ ۴ را بررسی کنید.')

try:
    validate(cfg)
    predictor = Predictor(CHECKPOINT, DEVICE)
    image, target, preprocessing = load_sample(selected)
    predicted_roi = predictor(image)
    digest, moments = roi_digest(image, predicted_roi)
    raw = stream(digest, image.size, cfg, raw=True)  # همان مصرف واقعی رمز: یک حالت به‌ازای هر پیکسل
    METHOD_READY = True
    print('پیش‌بررسی موفق؛ این هنوز اثبات پایداری یا امنیت نیست.')
    print('پروفایل:', cfg['profile'], '| سیستم:', cfg['system'])
    print('گشتاورهای ROI:', [round(m, 6) for m in moments])
    print('بازهٔ حالت‌های ODE پس از گذر:', float(raw.min()), 'تا', float(raw.max()))
except ValueError as exc:
    write_json(WORK/'method_preflight.json', {'status': 'blocked', 'error': f'{type(exc).__name__}: {exc}',
                                              'profile': cfg.get('profile'), 'system': cfg.get('system'),
                                              'parameters': cfg.get('parameters'), 'checkpoint': str(CHECKPOINT)})
    print('مانع ثبت‌شده:', exc)
    if cfg.get('system') == 'equation_3_2':
        print('راه پیش‌رو: (۱) عددهای رابطهٔ ۳-۲ را در پروفایل مستند خودتان تعیین و PROFILE را به آن اشاره دهید،')
        print('یا (۲) در همین سلول USE_APPENDIX_PROFILE=True کنید تا پروفایل مستند پیوست اجرا شود.')


## ۷. اجرای آزمایش‌ها
برای هر چهار مجموعه جدا اجرا کنید. پیش‌فرض ۱۰۰ تلاش تفاضلی در هر مجموعه است؛ نویز/برش، حساسیت digest، حذف مؤلفه‌ها و بازیابی دقیق هم ثبت می‌شوند. طول پیام و ظرفیت واقعی‌اند. اجرای همهٔ مجموعه‌ها ممکن است از زمان یک نشست کولب طولانی‌تر باشد.

In [ ]:
from scripts.experiments import run
from scripts.reporting import make_report
if RUN_EXPERIMENTS:
    if not METHOD_READY: raise RuntimeError('روش آماده نیست؛ خروجی بخش ۶ و WORK/method_preflight.json را ببینید')
    if not PAYLOAD.is_file(): raise FileNotFoundError('فایل فراداده آزمایشی لازم است: '+str(PAYLOAD))
    status = run(MANIFEST, DATASET, CHECKPOINT, cfg, PAYLOAD, RUN_DIR, DEVICE, include_ablations=True)
    print(status)
    print(make_report(RUN_DIR))
else:
    print('اجرای پژوهشی فعال نشده است؛ نتیجه‌ای ساخته نشد.')

## ۸. رفت‌وبرگشت مستقل تصویر و پیام
فایل راز خارج از گزارش و فایل‌های ارسالی نگه‌داری می‌شود. گیرنده به stego، recovery.msr و همان راز نیاز دارد. این افزونهٔ اصلاحی با ادعای گیرندهٔ بدون فایل کمکی پایان‌نامه متفاوت است.

In [ ]:
def cli(*args):
    return subprocess.run([sys.executable, '-m', 'scripts', *map(str,args)], check=True)
RUN_DELIVERY = False
if RUN_DELIVERY:
    if not METHOD_READY: raise RuntimeError('روش آماده نیست')
    KEYFILE = WORK/'private_research.key'
    if not KEYFILE.exists(): cli('keygen','--output',KEYFILE)
    SENT = WORK/'sent_001'; RECEIVED = WORK/'received_001'
    cli('send','--manifest',MANIFEST,'--dataset',DATASET,'--id',selected['id'],'--checkpoint',CHECKPOINT,
        '--config',PROFILE,'--payload',PAYLOAD,'--keyfile',KEYFILE,'--output',SENT,'--device',DEVICE)
    cli('receive','--stego',SENT/'stego.png','--sidecar',SENT/'recovery.msr','--keyfile',KEYFILE,'--output',RECEIVED)

## ۹. برآورد لیاپانوف و پرترهٔ فاز
این برآورد زمان محدود است. طول‌های متفاوت، چند شرایط اولیه و گام‌های کوچکتر برای بررسی همگرایی لازم‌اند؛ علامت مثبت به‌تنهایی اثبات نیست.

In [ ]:
from scripts.dynamics import lyapunov
if METHOD_READY and RUN_EXPERIMENTS:
    dynamics = lyapunov(digest, cfg, steps=100000, qr_interval=10, output=WORK/'dynamics'/f'{DATASET}.json')
    print({k:v for k,v in dynamics.items() if k not in ('phase','convergence','environment')})
    if dynamics['status'] == 'ok':
        import numpy as np
        phase = np.array(dynamics['phase'])
        fig = plt.figure(); ax = fig.add_subplot(111, projection='3d')
        ax.plot(phase[:,0], phase[:,1], phase[:,2], linewidth=.4)
        ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z'); plt.show()

## ۱۰. NIST رسمی: دریافت، ساخت و اجرا
این مرحله بدون دادهٔ کافی متوقف می‌شود؛ هیچ پرکردن/تکرار تصویر برای رسیدن به صد میلیون بیت انجام نمی‌شود. خروجی تمام ۱۵ خانواده و مؤلفه‌های آن‌ها در پوشهٔ نتیجه باقی می‌ماند. دادهٔ کمتر با برچسب آزمون نرم‌افزار از این مرحلهٔ پژوهشی جدا است.

In [ ]:
os.environ['NIST_STS_URL'] = 'https://csrc.nist.gov/CSRC/media/Projects/Random-Bit-Generation/documents/sts-2_1_2.zip'
if RUN_NIST:
    import urllib.request
    from scripts.nist import build_official, export_streams, run_official
    # فقط اجراهای ثبت‌شدهٔ کامل وارد می‌شوند؛ هیچ داده‌ای جعل یا تکرار نمی‌شود.
    NIST_RUNS = [p for p in (WORK/'runs'/f'{name}_001' for name in ['DRIVE', 'RITE', 'BraTS2020', 'COVID19_CXR'])
                 if (p/'run.json').is_file() and (p/'samples.jsonl').is_file()]
    if not NIST_RUNS:
        raise RuntimeError('هیچ اجرای ثبت‌شده‌ای برای NIST نیست؛ ابتدا بخش ۷ را برای مجموعه‌ها کامل کنید.')
    print('اجراهای واردشده به NIST:', [p.name for p in NIST_RUNS])
    archive = WORK/'sts-2_1_2.zip'
    if not archive.exists(): urllib.request.urlretrieve(os.environ['NIST_STS_URL'], archive)
    BUILD = WORK/'nist_build'
    if not BUILD.exists(): sts_root = build_official(archive, BUILD)
    else: sts_root = Path(json.loads((BUILD/'build.json').read_text())['assess']).parent
    nist_input = WORK/'nist_input_001'
    if not nist_input.exists(): export_streams(NIST_RUNS, nist_input, sequences=100, bits=1000000)
    nist_result = run_official(sts_root, nist_input, WORK/'nist_results_001', timeout=7200)
    print(nist_result['status'], 'families:', nist_result['families'])
else:
    print('آزمون رسمی NIST فعال نشده است (RUN_NIST=False).')


## ۱۱. جمع‌بندی و دریافت جدول‌ها
فقط گزارش اجرای انتخابی بسته‌بندی می‌شود؛ فایل راز و دادهٔ خام پزشکی در ZIP گزارش نیست. جدول literature_reported.csv نقل از پایان‌نامه است و بازاجرای مقاله‌ها نیست. زمان کولب را زمان Jetson معرفی نکنید.

In [ ]:
if (RUN_DIR/'status.json').exists():
    make_report(RUN_DIR)
    import shutil
    archive = shutil.make_archive(str(WORK/f'{DATASET}_report'), 'zip', RUN_DIR/'reports')
    files.download(archive)
else:
    print('هنوز اجرای ثبت‌شده‌ای برای گزارش وجود ندارد.')